# Member 1 — Data foundations (M1 / Lab 1)
**Phase 1 فقط.** تجهيز CCT20: فحص وتنظيف، تقسيم طبقي حسب الكاميرا، تدقيق، Preprocessor وDataLoader، ثم Dataset Card.
شغّل الخلايا بالتتابع؛ كل تنفيذ يعرض نتيجته. لا تدريب نموذج هنا. لا يتطلب Drive.
حجم العينة وعتبات التنظيف والتدقيق اختيارات الفريق، وليست تعليمات رقمية من الدكتور.
المخرجات تنتقل للعضوين 2 و3؛ احفظ النتائج قبل انتهاء جلسة Colab.

## 1. المكتبات والإعدادات — Lab 1
غيّر حجم العينة هنا قبل البدء. 6000 عدد تقريبي لأننا نحتفظ بكل صور الحدث معًا.

In [ ]:
%pip -q install ImageHash pandas scipy matplotlib Pillow PyYAML ipywidgets scikit-learn opencv-python-headless tqdm


In [ ]:
from pathlib import Path
import os, sys, subprocess
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
"""Download official CCT20 metadata; no predicted boxes are used as labels."""
import collections
import hashlib
import html
import io
import json
from pathlib import Path
import random
import tarfile
import time
import urllib.parse
import urllib.request
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
import imagehash


META_URL = 'https://storage.googleapis.com/public-datasets-lila/caltechcameratraps/eccv_18_annotations.tar.gz'
IMAGE_BASES = [
    'https://storage.googleapis.com/public-datasets-lila/caltech-unzipped/cct_images/',
    'https://lilawildlife.blob.core.windows.net/lila-wildlife/caltech-unzipped/cct_images/']
NON_ANIMALS = {'empty', 'car', 'vehicle', 'person', 'human', 'people'}



SEED = 42
MAX_IMAGES = 6000
AUDIT_IMAGES = 100
PROJECT = Path.cwd() if Path.cwd().name == 'CameraTrap_Phase1_Team' else Path.cwd()/'CameraTrap_Phase1_Team'
PROJECT.mkdir(exist_ok=True)
os.chdir(PROJECT)
DATA = Path('data/cct20_v1')
DATA.mkdir(parents=True, exist_ok=True)
Path('runs/data').mkdir(parents=True, exist_ok=True)
Path('handoff').mkdir(exist_ok=True)

def read_json(path):
    return json.loads(Path(path).read_text())
def write_json(path, value):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False))
def digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''): h.update(block)
    return h.hexdigest()

cfg = {'seed': SEED, 'max_images': MAX_IMAGES, 'max_image_side': 1024,
       'audit_events': AUDIT_IMAGES, 'data_dir': str(DATA), 'output_dir': 'runs/data'}
assert not Path('handoff/dataset_release.json').exists(), 'هذا الإصدار صدر بالفعل. استخدم نتائجه أو ابدأ مجلدًا جديدًا؛ لا تعدله بعد تدريب زملائك.'
print('Project folder:', PROJECT)


## 2. قراءة التسميات — Lab 1
نقرأ CCT20 ونستبعد التسميات البشرية/المركبات والمتعارضة. الصناديق بشرية وليست توقعات نموذج. تنزيل الصور في خطوة 5.

In [ ]:
def fetch(url, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.stat().st_size:
        return
    tmp = path.with_suffix(path.suffix+'.part')
    for attempt in range(3):
        try:
            with urllib.request.urlopen(url, timeout=45) as r, tmp.open('wb') as f:
                while chunk := r.read(1024*1024):
                    f.write(chunk)
            tmp.replace(path)
            return
        except Exception:
            tmp.unlink(missing_ok=True)
            if attempt == 2:
                raise
            time.sleep(attempt+1)

def read_metadata(root):
    archive = root/'metadata'/'eccv_18_annotations.tar.gz'
    fetch(META_URL, archive)
    images, annotations, categories = {}, {}, {}
    with tarfile.open(archive) as tar:
        for member in tar.getmembers():
            if not member.name.endswith('.json') or not member.isfile():
                continue
            
            d = json.load(tar.extractfile(member))
            for c in d['categories']:
                if c['id'] in categories and categories[c['id']] != c['name']:
                    raise ValueError('Conflicting category maps')
                categories[c['id']] = c['name']
            for im in d['images']:
                if im['id'] in images and images[im['id']] != im:
                    raise ValueError('Conflicting image metadata')
                images[im['id']] = im
            for a in d['annotations']:
                key = (a['image_id'], a['category_id'], tuple(a.get('bbox', [])))
                annotations[key] = a
    by_image = collections.defaultdict(list)
    for a in annotations.values():
        by_image[a['image_id']].append(a)
    rows, excluded = [], []
    for ident, im in sorted(images.items()):
        aa = by_image[ident]
        labels = sorted({categories[a['category_id']] for a in aa})
        if not labels or any(c in NON_ANIMALS-{'empty'} for c in labels):
            excluded.append({'id': ident, 'reason': 'non-animal-category-or-unlabelled', 'labels': labels})
            continue
        animal_labels = [s for s in labels if s not in NON_ANIMALS]
        if 'empty' in labels and animal_labels:
            excluded.append({'id': ident, 'reason': 'conflicting-empty-label'})
            continue
        boxes = []
        incomplete = False
        for a in aa:
            name = categories[a['category_id']]
            if name in NON_ANIMALS:
                continue
            if 'bbox' not in a:
                incomplete = True
                continue
            x, y, w, h = map(float, a['bbox'])
            W, H = im['width'], im['height']
            x1, y1 = max(0.,x/W), max(0.,y/H)
            x2, y2 = min(1.,(x+w)/W), min(1.,(y+h)/H)
            if x2 <= x1 or y2 <= y1:
                incomplete = True
                continue
            boxes.append({'species': name, 'xyxy': [x1,y1,x2,y2], 'annotation_source': 'human_CCT20'})
        rows.append(dict(id=ident, file_name=Path(im['file_name']).name,
                         camera=str(im['location']), event=str(im.get('seq_id') or ident),
                         timestamp=im.get('date_captured',im.get('datetime')),
                         original_width=im['width'], original_height=im['height'],
                         seq_num_frames=im.get('seq_num_frames'), labels=animal_labels,
                         animal=int(bool(animal_labels)), boxes=boxes,
                         incomplete_boxes=incomplete))
    summary = dict(source=META_URL, sha256=digest(archive), total_images=len(images),
                   unique_annotations=len(annotations), human_boxes=sum('bbox' in a for a in annotations.values()),
                   cameras=len({str(i['location']) for i in images.values()}),
                   categories=categories, eligible_frames=len(rows), excluded_frames=len(excluded),
                   empty_frames=sum(r['animal']==0 for r in rows),
                   note='CCT20 is curated and is not the full 243k-image dataset. Percent empty is recomputed.')
    write_json(root/'metadata_summary.json', summary)
    write_json(root/'metadata_exclusions.json', excluded)
    return rows

rows = read_metadata(DATA)
print("Eligible frames:", len(rows))
display(pd.DataFrame(rows)[["id", "camera", "event", "animal", "labels"]].head())

## 3. Stratified Group Split — Lab 1 / Part C
نستخدم StratifiedGroupKFold مثل اللاب، لكن group = موقع الكاميرا. نحاول تقريب نسب تسميات الصور مع إبقاء الكاميرات وbursts منفصلة. هدف تقريبي: 60% train و10% val_select و10% val_calibrate و20% test؛ حجم المجموعات الفعلي يتأثر بالكاميرات. هذا تقسيم مخصص وليس benchmark الرسمي. الفئات النادرة قد تغيب عن بعض المجموعات؛ سنعرض ذلك صراحة.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

def assign_sites(rows, seed):
    
    y=np.array(['|'.join(sorted(r['labels'])) or 'empty' for r in rows])
    groups=np.array([r['camera'] for r in rows])
    def first_fold(indices, folds):
        splitter=StratifiedGroupKFold(n_splits=folds,shuffle=True,random_state=seed)
        a,b=next(splitter.split(np.zeros(len(indices)),y[indices],groups[indices]))
        return indices[a],indices[b]
    remaining,test=first_fold(np.arange(len(rows)),5)
    train,validation=first_fold(remaining,4)
    select,calibrate=first_fold(validation,2)
    site_map={}
    for name,ids in [('train',train),('val_select',select),('val_calibrate',calibrate),('test',test)]:
        for camera in set(groups[ids]):
            assert camera not in site_map
            site_map[camera]=name
    return site_map

def assert_disjoint(rows):
    for field in ['id', 'camera', 'event', 'pixel_sha256']:
        seen = {}
        for r in rows:
            if field not in r:
                continue
            key = r[field]
            if key in seen and seen[key] != r['split']:
                raise AssertionError(f'{field} leakage across splits')
            seen[key] = r['split']

site_map=assign_sites(rows,SEED)
for r in rows:r['split']=site_map[r['camera']]
assert_disjoint(rows)
write_json(DATA/'site_split.json',site_map)
display(pd.DataFrame(rows).groupby('split').agg(images=('id','count'),cameras=('camera','nunique')))


## 4. اختيار العينة مع الحفاظ على bursts
نختار أحداثًا كاملة من كل كاميرا، حتى لا تتوزع صور الحدث على أكثر من split.

In [ ]:
def sample_events(rows, limit, seed):
    ""
    if not limit or limit >= len(rows):
        return rows
    rng = random.Random(seed)
    selected = []
    for camera in sorted({r['camera'] for r in rows}):
        rr = [r for r in rows if r['camera']==camera]
        groups = collections.defaultdict(list)
        for r in rr:
            groups[r['event']].append(r)
        events = sorted(groups)
        rng.shuffle(events)
        budget = max(3, round(limit*len(rr)/len(rows)))
        chosen = []
        for event in events:
            chosen.extend(groups[event])
            if len(chosen) >= budget:
                break
        selected.extend(chosen)
    return selected

chosen = sample_events(rows, MAX_IMAGES, SEED)
write_json(DATA/'selected_ids.json', [r['id'] for r in chosen])
print('Selected images:', len(chosen))


## 5. تنزيل الصور وفحص الملفات — Lab 1
نصغّر جانب الصورة الأكبر إلى 1024 لتقليل الحجم. الملفات غير المقروءة تُسجّل، ونستبعد أحداثها كاملة. إعادة هذه الخلية تعيد محاولة الملفات التي فشلت.

In [ ]:
def download_one(row, root, max_side):
    path = root/'images'/(row['id']+'.jpg')
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        err = None
        for base in IMAGE_BASES:
            try:
                with urllib.request.urlopen(base+urllib.parse.quote(row['file_name']),timeout=40) as r:
                    raw = r.read()
                im = Image.open(io.BytesIO(raw)).convert('RGB')
                im.thumbnail((max_side,max_side), Image.Resampling.LANCZOS)
                tmp = path.with_suffix('.tmp')
                im.save(tmp, format='JPEG',quality=95)
                tmp.replace(path)
                break
            except Exception as e:
                err = str(e)
        else:
            return None, {'id':row['id'],'event':row['event'],'reason':err}
    try:
        with Image.open(path) as src:
            im = src.convert('RGB'); im.load()
            small = np.array(im.resize((64,64)),dtype=float)
            row = dict(row, path=str(path.resolve()),width=im.width,height=im.height,
                       processed_file_md5=hashlib.md5(path.read_bytes()).hexdigest(),
                       pixel_sha256=hashlib.sha256(im.tobytes()).hexdigest(),
                       phash=str(imagehash.phash(im)),
                       grayscale_proxy=bool(np.mean(np.max(small,2)-np.min(small,2))<6))
            return row, None
    except Exception as e:
        return None, {'id':row['id'],'event':row['event'],'reason':str(e)}

from tqdm.auto import tqdm
good, failures = [], []
for row in tqdm(chosen):
    image_row, error = download_one(row, DATA, cfg['max_image_side'])
    if image_row is not None:
        image_row['path'] = str(DATA/'images'/(image_row['id']+'.jpg'))  
        good.append(image_row)
    if error is not None: failures.append(error)
write_json(DATA/'download_failures.json', failures)
print('Readable:', len(good), '| Failed:', len(failures))
assert len(failures) <= 0.02*len(chosen), 'فشل أكثر من 2%؛ أعد خلية التنزيل قبل المتابعة'
failed_events = {e['event'] for e in failures}
good = sorted([r for r in good if r['event'] not in failed_events], key=lambda r:r['id'])


### فحص الصور والألوان والجودة — Lab 1 / A وD.2
نعرض الأبعاد والألوان لنسخ الصور المنزلة، ونقيس الضبابية بعد resize إلى 224. نراجع الأقل حدة من train فقط؛ الخلفية البسيطة قد تعطي درجة منخفضة رغم صلاحية الصورة.

In [ ]:
import cv2
inventory=[]
for r in good:
    im=Image.open(r['path']);rgb=np.asarray(im.convert('RGB'))
    small=cv2.resize(rgb,(224,224),interpolation=cv2.INTER_AREA)
    blur=float(cv2.Laplacian(cv2.cvtColor(small,cv2.COLOR_RGB2GRAY),cv2.CV_64F).var())
    inventory.append({'id':r['id'],'split':r['split'],'mode':im.mode,'width':im.width,'height':im.height,
      'brightness':float(small.mean()),'blur_score':blur,'path':r['path']})
inventory=pd.DataFrame(inventory);inventory.to_csv(DATA/'inventory.csv',index=False)
display(inventory.describe(include='all'))
lowest=inventory.query("split == 'train'").nsmallest(8,'blur_score').copy()
lowest['reviewer']='';lowest['decision']='';lowest['notes']=''
if not (DATA/'blur_review.csv').exists():lowest.to_csv(DATA/'blur_review.csv',index=False)
for r in lowest.to_dict('records'):
    print(r['id'],'blur:',round(r['blur_score'],2));im=Image.open(r['path']);im.thumbnail((420,280));display(im)
print('أكمل blur_review.csv: reviewer = اسمك، decision = keep أو flag، notes = السبب. flag للتوثيق فقط؛ لا حذف تلقائي.')


## 6. إزالة التكرار وفحص التسرب — Lab 1
نزيل التكرار المطابق. عند التكرار بين splits نستبعد الحدثين. التشابه الشديد بـpHash إجراء تحفظي؛ لا يعني أن كل زوج متشابه نسخة مؤكدة.

In [ ]:
def near_duplicate_quarantine(rows, distance=2):
    
    buckets = collections.defaultdict(list)
    bad, pairs = set(), []
    for r in rows:
        h = int(r['phash'],16)
        candidates = {}
        for part in range(4):
            for old in buckets[(part,(h>>(part*16)) & 65535)]:
                candidates[old['id']] = old
        for old in candidates.values():
            if old['split']==r['split']:
                continue
            dist = (h ^ int(old['phash'],16)).bit_count()
            if dist <= distance:
                bad.update([r['event'],old['event']])
                pairs.append({'id_a':old['id'],'id_b':r['id'],'distance':dist})
        for part in range(4):
            buckets[(part,(h>>(part*16)) & 65535)].append(r)
    return [r for r in rows if r['event'] not in bad], pairs, sorted(bad)

In [ ]:
hashes=collections.defaultdict(list)
for r in good:hashes[r['pixel_sha256']].append(r)
excluded_events=set(); duplicate_ids=[];conflicting_duplicate_groups=[]
for group in hashes.values():
    if len(group)>1:
        if len({tuple(r['labels']) for r in group})>1:
            conflicting_duplicate_groups.append([r['id'] for r in group])
            excluded_events.update(r['event'] for r in group)
        elif len({r['split'] for r in group})>1:
            excluded_events.update(r['event'] for r in group)
        else:duplicate_ids.extend(r['id'] for r in group[1:])
cleaned=[r for r in good if r['event'] not in excluded_events and r['id'] not in set(duplicate_ids)]
cleaned,pairs,near_events=near_duplicate_quarantine(cleaned)
assert_disjoint(cleaned)
for split in ['train','val_select','val_calibrate','test']:
    if {r['animal'] for r in cleaned if r['split']==split}!={0,1}:
        raise ValueError(f'{split} lacks empty/animal examples after sampling. Increase max_images; do not adjust test based on results.')
labels=sorted({b['species'] for r in cleaned if r['split']=='train' for b in r['boxes']})
if len(labels)<2:raise ValueError('Need at least two training species')
write_json(DATA/'species.json',labels)
write_json(DATA/'manifest.json',cleaned)
write_json(DATA/'preparation_config.json',cfg)
write_json(DATA/'cleaning_report.json',dict(selected=len(chosen),downloaded=len(good),kept=len(cleaned),
    failures=len(failures),excluded_failed_events=len(failed_events),duplicate_ids=duplicate_ids,
    exact_cross_split_events=sorted(excluded_events),near_duplicate_pairs=pairs,near_quarantined_events=near_events,
    conflicting_duplicate_label_groups=conflicting_duplicate_groups,
    near_policy='Conservative pHash <=2 across partitions: both events quarantined; not asserted to be true duplicates.',
    corrupted_or_failed_files=failures,grayscale_proxy='RGB channel difference <6; proxy only, not verified infrared ground truth'))

print('Clean frames:', len(cleaned), '| Training species:', labels)

## 7. عرض الصور وتوزيع الفئات — Lab 1
نراجع مثالًا من train ونرسم التوزيع لكل split. labels على الصور الكاملة، وspecies على القصاصات البشرية.

In [ ]:
def gallery(path, rows, draw_boxes=False, titles=None):
    path=Path(path);thumbs=path.parent/(path.stem+'_images');thumbs.mkdir(exist_ok=True)
    cards=[]
    for i,r in enumerate(rows):
        im=Image.open(r['path']).convert('RGB')
        if draw_boxes:
            draw=ImageDraw.Draw(im)
            for b in r['boxes']:
                x1,y1,x2,y2=b['xyxy'];box=[x1*im.width,y1*im.height,x2*im.width,y2*im.height]
                draw.rectangle(box,outline='lime',width=3);draw.text((box[0],box[1]),b['species'],fill='yellow')
        im.thumbnail((480,360));name=f'{i:04d}.jpg';im.save(thumbs/name)
        title=titles[i] if titles else f'{r["id"]} | {r["split"]} | {r["labels"] or ["empty"]}'
        cards.append(f'<article><img src="{thumbs.name}/{name}"><p>{html.escape(str(title))}</p></article>')
    path.write_text('<!doctype html><meta charset="utf-8"><title>Local review gallery</title><style>body{font:14px sans-serif;background:#132127;color:white}main{display:grid;grid-template-columns:repeat(auto-fit,minmax(340px,1fr));gap:16px}article{background:#22343b;padding:10px}img{max-width:100%}p{overflow-wrap:anywhere}</style><h1>Private local review gallery</h1><p>Review before sharing; exclude or blur any identifiable people. Do not redistribute dataset images.</p><main>'+''.join(cards)+'</main>',encoding='utf-8')

def audit_templates(cfg, rows):
    root=Path(cfg['data_dir'])
    
    rng=random.Random(cfg['seed'])
    events=collections.defaultdict(list)
    for r in rows:
        if r['split']=='train':events[r['event']].append(r)
    selected=rng.sample(sorted(events),min(cfg['audit_events'],len(events)))
    audit=[]
    for event in selected:
        r=rng.choice(events[event]);audit.append(dict(id=r['id'],event=event,split=r['split'],
          expected_species=';'.join(r['labels']) or 'empty',label_correct='',boxes_correct='',
          identifiable_person='',reviewer='',notes=''))
    pd.DataFrame(audit).to_csv(root/'label_audit.csv',index=False)
    gallery(root/'audit_gallery.html',[next(r for r in rows if r['id']==a['id']) for a in audit],draw_boxes=True)
    
    count_rows=[dict(split=r['split'],label=s) for r in rows for s in (r['labels'] or ['empty'])]
    pd.crosstab(pd.DataFrame(count_rows).label,pd.DataFrame(count_rows).split).to_csv(root/'class_counts.csv')
    crop_rows=[dict(split=r['split'],label=b['species']) for r in rows for b in r['boxes']]
    pd.crosstab(pd.DataFrame(crop_rows).label,pd.DataFrame(crop_rows).split).to_csv(root/'crop_counts.csv')
    gallery(root/'annotation_gallery.html',rng.sample(rows,min(48,len(rows))),draw_boxes=True)

if not (DATA/'label_audit.csv').exists():
    audit_templates(cfg, cleaned)
counts = pd.read_csv(DATA/'class_counts.csv', index_col=0)
display(counts)
counts.plot.bar(stacked=True, figsize=(12,4))
plt.ylabel('Label occurrences'); plt.tight_layout(); plt.show()
for row in [r for r in cleaned if r['split']=='train'][:4]:
    print(row['id'], row['labels'] or ['empty'])
    im=Image.open(row['path']); im.thumbnail((600,400)); display(im)


## 8. تدقيق بشري — Lab 1
راجع العينات، اختر هل التسمية والصناديق صحيحة وهل يظهر شخص، واكتب اسمك ثم Save. أكمل جميع العينات قبل المتابعة. لا تعتبر الخانة الفارغة موافقة.

In [ ]:
def audit_widget(cfg):
    import ipywidgets as w
    from IPython.display import display,clear_output
    from PIL import Image,ImageDraw
    root=Path(cfg['data_dir']);path=root/'label_audit.csv'
    df=pd.read_csv(path).fillna('');rows={r['id']:r for r in read_json(root/'manifest.json')}
    index=w.IntSlider(value=0,min=0,max=len(df)-1,description='Sample')
    correct=w.Dropdown(options=['','yes','no'],description='Label correct')
    boxes=w.Dropdown(options=['','yes','no','not applicable'],description='Boxes correct')
    person=w.Dropdown(options=['','yes','no'],description='Person visible')
    reviewer=w.Text(description='Reviewer');notes=w.Textarea(description='Notes')
    save=w.Button(description='Save and next',button_style='success');output=w.Output();status=w.Output()
    def show(change=None):
        row=df.iloc[index.value];r=rows[row.id]
        correct.value=str(row.label_correct);boxes.value=str(row.boxes_correct);person.value=str(row.identifiable_person)
        if row.reviewer:reviewer.value=str(row.reviewer)
        notes.value=str(row.notes)
        with output:
            clear_output(wait=True);print('Expected:',row.expected_species,'| sample:',row.id)
            im=Image.open(r['path']).convert('RGB');draw=ImageDraw.Draw(im)
            for b in r['boxes']:
                x1,y1,x2,y2=b['xyxy'];draw.rectangle([x1*im.width,y1*im.height,x2*im.width,y2*im.height],outline='lime',width=3)
            im.thumbnail((900,600));display(im)
    def persist(_):
        with status:
            clear_output()
            if not reviewer.value.strip() or not correct.value or not person.value or not boxes.value:
                print('Fill every decision and the reviewer name before saving.');return
        i=index.value
        for key,value in [('label_correct',correct.value),('boxes_correct',boxes.value),('identifiable_person',person.value),
                          ('reviewer',reviewer.value.strip()),('notes',notes.value)]:df.loc[i,key]=value
        df.to_csv(path,index=False)
        with status:print('Saved. Corrections/privacy flags need resolution before final training/report.')
        if i<len(df)-1:index.value=i+1
    index.observe(show,names='value');save.on_click(persist)
    display(w.VBox([index,output,correct,boxes,person,reviewer,notes,save,status]));show()

audit_widget(cfg)

## 9. معالجة مشاكل التدقيق وحساب حد الخطأ
نستبعد أحداث train التي علّمها المراجع كمشكلات، ونحفظ قراراته. حد الخطأ محسوب من العينة الأصلية؛ افتراض استقلال الأحداث تقريبي.

In [ ]:
from scipy.stats import beta

def audit_report(cfg):
    root=Path(cfg['data_dir']);df=pd.read_csv(root/'label_audit.csv').fillna('')
    valid=(df.label_correct.astype(str).str.lower().isin(['yes','no']) &
           df.boxes_correct.astype(str).str.lower().isin(['yes','no','not applicable']) &
           df.identifiable_person.astype(str).str.lower().isin(['yes','no']) &
           df.reviewer.astype(str).str.strip().ne(''))
    reviewed=df[valid];n=len(reviewed);errors=int((reviewed.label_correct.astype(str).str.lower()=='no').sum())
    rows=read_json(root/'manifest.json');present_events={r['event'] for r in rows}
    flagged=reviewed[(reviewed.label_correct.astype(str).str.lower()=='no') |
                     (reviewed.boxes_correct.astype(str).str.lower()=='no') |
                     (reviewed.identifiable_person.astype(str).str.lower()=='yes')]
    upper=float(beta.ppf(.95,errors+1,n-errors)) if n and errors<n else (1. if n else None)
    result=dict(status='complete' if n==len(df) else 'pending',requested=len(df),reviewed=n,label_errors=errors,
      estimated_error_rate=errors/n if n else None,one_sided_95_upper_bound=upper,
      sampling_unit='random train event, then one random eligible frame; not all dataset frames',
      limitations='Binomial interval assumes independent sampled events. Same-camera correlation may make it optimistic.',
      no_review_is_not_zero_errors=True,
      privacy_flags=int(((reviewed.identifiable_person.astype(str).str.lower()=='yes') & reviewed.event.isin(present_events)).sum()),
      unresolved_flagged_events=len(set(flagged.event)&present_events))
    write_json(root/'audit_report.json',result);return result

def apply_audit(cfg):
    ""
    root=Path(cfg['data_dir']);out=Path(cfg['output_dir'])
    df=pd.read_csv(root/'label_audit.csv').fillna('')
    reviewed=df[df.reviewer.astype(str).str.strip().ne('')]
    flagged=reviewed[(reviewed.label_correct.astype(str).str.lower()=='no') |
                     (reviewed.boxes_correct.astype(str).str.lower()=='no') |
                     (reviewed.identifiable_person.astype(str).str.lower()=='yes')]
    bad=set(flagged.event);rows=read_json(root/'manifest.json')
    if not (bad & {r['event'] for r in rows}):
        print('No unresolved human-flagged events to quarantine.');return
    if list(out.glob('*/*/best.pt')):
        raise RuntimeError('Checkpoints already exist. Set a NEW output_dir in config before applying training-data corrections.')
    if any(r['split']!='train' and r['event'] in bad for r in rows):
        raise ValueError('Only train audit changes are allowed; held-out protocol is immutable.')
    cleaned=[r for r in rows if r['event'] not in bad]
    labels=sorted({b['species'] for r in cleaned if r['split']=='train' for b in r['boxes']})
    if len(labels)<2 or {r['animal'] for r in cleaned if r['split']=='train'}!={0,1}:
        raise ValueError('Audit exclusions remove required training support; collect more training events.')
    write_json(root/'audit_quarantine.json',dict(events=sorted(bad),frames_removed=len(rows)-len(cleaned),
        policy='Human-flagged events excluded. No automatic corrected species labels.'))
    write_json(root/'manifest.json',cleaned);write_json(root/'species.json',labels);assert_disjoint(cleaned)
    for name,values in [('class_counts',[dict(split=r['split'],label=s) for r in cleaned for s in (r['labels'] or ['empty'])]),
                        ('crop_counts',[dict(split=r['split'],label=b['species']) for r in cleaned for b in r['boxes']])]:
        frame=pd.DataFrame(values);pd.crosstab(frame.label,frame.split).to_csv(root/(name+'.csv'))
    print('Quarantined',len(rows)-len(cleaned),'training frames across',len(bad),'flagged events.')

audit = audit_report(cfg)
assert audit['status']=='complete', 'أكمل نموذج التدقيق أولاً'
apply_audit(cfg)
audit = audit_report(cfg)
assert audit['unresolved_flagged_events']==0
rows = read_json(DATA/'manifest.json')
print(audit)


### التحقق النهائي من Class Stratification — Lab 1 / C وD.1
هذه الأرقام بعد التنظيف والتدقيق. لا تعني stratification أن الفئات تصبح متساوية؛ نحاول الحفاظ على نسبها بين splits. نعرض فرق النسبة بالنقاط المئوية وغياب الأنواع، ولا نغيّر التقسيم بناءً على دقة Test.

In [ ]:
splits=['train','val_select','val_calibrate','test']
all_labels=sorted({label for r in rows for label in (r['labels'] or ['empty'])})
balance=[]
for label in all_labels:
    overall=sum(label in (r['labels'] or ['empty']) for r in rows)/len(rows)
    for split in splits:
        subset=[r for r in rows if r['split']==split]
        count=sum(label in (r['labels'] or ['empty']) for r in subset)
        rate=count/len(subset) if subset else 0
        balance.append({'label':label,'split':split,'count':count,'frames':len(subset),
                        'percent':100*rate,'overall_percent':100*overall,'difference_pp':100*(rate-overall)})
balance_report=pd.DataFrame(balance);balance_report.to_csv(DATA/'class_balance_proportions.csv',index=False)
display(balance_report)
balance_report.pivot(index='label',columns='split',values='percent').plot.bar(figsize=(12,5))
plt.ylabel('Frames containing label (%)');plt.tight_layout();plt.show()
counts=pd.DataFrame([{'species':b['species'],'split':r['split']} for r in rows for b in r['boxes']])
support=pd.crosstab(counts.species,counts.split).reindex(columns=splits,fill_value=0)
support.to_csv(DATA/'species_crop_support.csv');display(support)
print('Missing class/split combinations:');display(balance_report.query('count == 0')[['label','split']])
assert_disjoint(rows)
print('PASS: cameras, events and exact pixel duplicates are disjoint.')


## 10. Preprocessor + Dataset + DataLoader — Lab 1 / B,E,F
نفس الكود مستخدم في دفتري التدريب: OpenCV BGR→RGB، resize إلى 224 (AREA للتصغير وLINEAR للتكبير)، augmentation على train، ثم normalization وCHW. معاينة فقط؛ لا تدريب.

In [ ]:
import cv2, numpy as np, torch
from PIL import Image
from torchvision import transforms
from torch.utils.data import DataLoader
class Preprocessor:
    ""
    def __init__(self,size):
        self.size=size
        self.mean=np.array([.485,.456,.406],np.float32)
        self.std=np.array([.229,.224,.225],np.float32)
    def prepare(self,bgr):
        rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB);h,w=rgb.shape[:2]
        interp=cv2.INTER_AREA if h>self.size or w>self.size else cv2.INTER_LINEAR
        return cv2.resize(rgb,(self.size,self.size),interpolation=interp)
    def finalize(self,rgb):return (rgb.astype(np.float32)/255.-self.mean)/self.std

def load_rgb_sample(row):
    im=cv2.imread(row['path'])
    if im is None:raise ValueError('Unreadable image: '+row['path'])
    if row.get('crop') is not None:
        x1,y1,x2,y2=row['crop'];h,w=im.shape[:2];dx=(x2-x1)*.05;dy=(y2-y1)*.05
        left,top=max(0,int((x1-dx)*w)),max(0,int((y1-dy)*h))
        right,bottom=min(w,max(left+1,int((x2+dx)*w))),min(h,max(top+1,int((y2+dy)*h)))
        im=im[top:bottom,left:right].copy()
    return im

class FrameDataset(torch.utils.data.Dataset):
    ""
    def __init__(self,samples,size,train=False):
        self.samples=samples;self.pre=Preprocessor(size)
        self.tf=transforms.Compose([transforms.RandomHorizontalFlip(),
                  transforms.ColorJitter(brightness=.15,contrast=.15)]) if train else None
    def __len__(self):return len(self.samples)
    def __getitem__(self,i):
        row=self.samples[i];img=self.pre.prepare(load_rgb_sample(row))
        if self.tf:img=np.array(self.tf(Image.fromarray(img)))
        x=torch.from_numpy(np.ascontiguousarray(self.pre.finalize(img))).permute(2,0,1).float()
        return x,int(row['label']),i

train_rows=[dict(r,label=r['animal'],crop=None) for r in rows if r['split']=='train']
train_dataset=FrameDataset(train_rows,224,train=True)
eval_dataset=FrameDataset(train_rows,224,train=False)
train_loader=DataLoader(train_dataset,batch_size=8,shuffle=True,generator=torch.Generator().manual_seed(SEED))
images,targets,_=next(iter(train_loader))
print('N,C,H,W:',images.shape,'| labels:',targets.tolist())
fig,axes=plt.subplots(2,4,figsize=(12,6))
for x,y,ax in zip(images,targets,axes.flat):
    rgb=x.permute(1,2,0).numpy()*[.229,.224,.225]+[.485,.456,.406]
    ax.imshow(np.clip(rgb,0,1));ax.set_title(['empty','animal'][int(y)]);ax.axis('off')
plt.tight_layout();plt.show()


### قبل وبعد — Lab 1 / A.5 وB
الصورة الأصلية هنا هي النسخة المنزلة بحد أقصى 1024. مقارنة RGB قبل resize، وبعد prepare، وبعد augmentation. الهستوغرام يصف الصورة والخلفية، ولا يثبت وجود الحيوان.

In [ ]:
sample=next(r for r in rows if r['split']=='train' and r['animal'])
bgr=cv2.imread(sample['path']);pre=Preprocessor(224)
rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB);prepared=pre.prepare(bgr)
augmented=np.array(train_dataset.tf(Image.fromarray(prepared)))
fig,axes=plt.subplots(1,3,figsize=(12,4))
for ax,img,title in zip(axes,[rgb,prepared,augmented],['Downloaded RGB','Resize 224','Train augmentation']):
    ax.imshow(img);ax.set_title(title);ax.axis('off')
plt.show()
for ch,color in enumerate(['red','green','blue']):
    plt.hist(prepared[:,:,ch].ravel(),bins=32,alpha=.35,color=color,label=color)
plt.legend();plt.title('Train sample channel histogram');plt.show()


## 11. مثال قصاصة لتصنيف النوع
نستخدم الصندوق البشري الموجود مع البيانات ونضيف سياقًا 5% حوله، مثل Member 2. لا ندرب detector.

In [ ]:
row = next(r for r in rows if r['split']=='train' and r['boxes'])
box = row['boxes'][0]
im = Image.open(row['path']).convert('RGB')
x1,y1,x2,y2 = box['xyxy']
dx,dy=(x2-x1)*.05,(y2-y1)*.05
crop=im.crop((max(0,int((x1-dx)*im.width)),max(0,int((y1-dy)*im.height)),
              min(im.width,int((x2+dx)*im.width)),min(im.height,int((y2+dy)*im.height))))
print('Species:', box['species']); display(crop)


## 12. Dataset Card — مخرج مطلوب
نكتب المصدر والترخيص والتنظيف والتقسيم والتوزيع والتدقيق والقيود. أضف ملاحظاتك الحقيقية في المتغير أدناه.

In [ ]:
TEAM_NOTES = ""  
counts = pd.read_csv(DATA/'class_counts.csv', index_col=0)
cleaning = read_json(DATA/'cleaning_report.json')
summary = pd.DataFrame(rows).groupby('split').agg(frames=('id','count'),cameras=('camera','nunique'))
card = f"""# Dataset Card — CCT20 / Phase 1
Source: https://lila.science/datasets/caltech-camera-traps
Citation: Beery, Van Horn and Perona, Recognition in Terra Incognita, ECCV 2018.
Licence: CDLA-Permissive; follow course policy: do not redistribute images or commit them to Git.

## Tasks
Full-frame empty/animal classification; species classification on human-box crops.
No predicted boxes are used as ground truth. Positive frames without boxes remain in triage only.
Human/vehicle-labelled, conflicting or unlabelled frames are excluded by metadata.

## Acquisition and cleaning
Selected: {cleaning['selected']}; retained after human audit: {len(rows)}.
Download failures: {cleaning['failures']}; exact duplicate IDs removed: {len(cleaning['duplicate_ids'])}.
Near-duplicate pairs across splits: {len(cleaning['near_duplicate_pairs'])}.
Image maximum side: 1024; model input: 224 x 224; seed: {SEED}.
Corrupt/failed-event and duplicate exclusions are recorded in cleaning_report.json.

## Split
StratifiedGroupKFold by camera, with joint image-label strata. Camera counts and class proportions below are measured, not assumed.
Whole bursts stay within one split; exact duplicate and event leakage checks pass.
Group constraints approximate class stratification; this is not the official CCT benchmark split.
{summary.to_string()}

## Long-tailed class distribution (frame label occurrences)
{counts.to_string()}
Species crop counts: crop_counts.csv. Rare species may have little or zero support in held-out locations.
Training vocabulary is built from train crops only; unseen held-out species are reported by Member 2.

## Final stratification check
{balance_report.to_string(index=False)}
Rates count label presence per frame; multi-label class percentages need not sum to 100.
See class_balance_proportions.csv and species_crop_support.csv. Camera constraints can prevent exact balance.

## Label audit
Reviewed: {audit['reviewed']} / {audit['requested']}; observed label errors: {audit['label_errors']}.
One-sided 95% upper error bound: {audit['one_sided_95_upper_bound']}.
Sampling: one frame per randomly sampled train event. Same-camera dependence may make this bound optimistic.
Flagged training events are quarantined; original audit decisions remain recorded.

## Preprocessing
RGB, resize 224, ImageNet normalization. Random flip and brightness/contrast changes on train only.
The identical OpenCV Preprocessor and augmentation code is embedded in Members 1, 2 and 3.

## Limitations and privacy
Curated US wildlife data; no claim of Saudi generalisation. The empty fraction must be measured on this subset.
Grayscale is an infrared proxy, not verified nighttime. Blur is a review flag; no image is discarded solely for low Laplacian variance. No GPS publication; keep photos and review galleries private.
Test accuracy confidence intervals belong to Member 2 after training, not this untrained data notebook.

## Team observations
{TEAM_NOTES or 'Pending: add actual observations before submission.'}
"""
Path('runs/data/DATASET_CARD.md').write_text(card)
display(Markdown(card))


## 13. حفظ ملفات التسليم للعضوين 2 و3
هذه خلية توافق مع دفاتر الفريق: تحفظ الإعدادات وقائمة البيانات وتثبت إصدارها. لا تحتاج تعديلها. بعد هذه الخلية لا تغيّر الصور أو التدقيق لنفس الإصدار.

In [ ]:
blur_check=pd.read_csv(DATA/'blur_review.csv').fillna('')
assert len(blur_check) and blur_check['reviewer'].str.strip().ne('').all() and blur_check['decision'].isin(['keep','flag']).all() and blur_check['notes'].str.strip().ne('').all(), 'أكمل blur_review.csv: reviewer وdecision = keep أو flag؛ flag للتوثيق ولا يحذف الصورة'
assert TEAM_NOTES.strip(), 'أضف ملاحظاتك الفعلية في Dataset Card وأعد تشغيل خليتها'
training_defaults = {'project': 'camera-trap-phase1', 'backbone': 'microsoft/resnet-50', 'seed': 42, 'profile': 'full', 'data_dir': 'data/cct20_v1', 'output_dir': 'runs/data', 'max_images': 6000, 'download_workers': 12, 'max_image_side': 1024, 'image_size': 224, 'batch_size': 32, 'workers': 2, 'epochs': 12, 'patience': 4, 'warmup_epochs': 2, 'lr_scratch': 0.001, 'lr_frozen': 0.001, 'lr_full': 0.0001, 'weight_decay': 0.0001, 'label_smoothing': 0.05, 'class_weighted_loss': True, 'wandb_mode': 'offline', 'target_animal_recall': 0.99, 'species_review_confidence': 0.6, 'bootstrap_repeats': 1000, 'audit_events': 100, 'acceptance': {'animal_recall_min': 0.99, 'empty_removal_min': 0.2, 'species_macro_f1_min': 0.65, 'species_ece_max': 0.1}}
training_defaults.update(cfg)
training_defaults.update(contract_version=2,preprocessing_sha256='87e3366dd7946e566c3a9238095748eb698ed0b57e2a9a5699d49817c440a5eb',warmup_epochs=0,weight_decay=0.0,label_smoothing=0.0,class_weighted_loss=False,optimizer='Adam',wandb_mode='offline')
write_json('handoff/preprocessing.json',{'sha256':'87e3366dd7946e566c3a9238095748eb698ed0b57e2a9a5699d49817c440a5eb','input_size':224,'library':'OpenCV + torchvision','train_only':['horizontal_flip','brightness_contrast']})
write_json('handoff/data_config.json', training_defaults)
write_json(DATA/'preparation_config.json', training_defaults)
Path('handoff/requirements_member1.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
write_json('handoff/runtime_member1.json', {'python':sys.version, 'platform':sys.platform})
files=[DATA/name for name in ['manifest.json','species.json','site_split.json','audit_report.json',
 'label_audit.csv','preparation_config.json','cleaning_report.json','class_counts.csv','crop_counts.csv','metadata_summary.json','class_balance_proportions.csv','species_crop_support.csv','inventory.csv','blur_review.csv']]
files += [Path('handoff/data_config.json'),Path('runs/data/DATASET_CARD.md'),Path('handoff/preprocessing.json')]
write_json('handoff/dataset_release.json', {'schema':2,'scope':'Phase 1 M1-M3',
 'data_dir':str(DATA),'audit_complete':True,'files':{str(f):digest(f) for f in files},
 'owner':'member1','source':'CCT20 human annotations','dataset_card':'runs/data/DATASET_CARD.md'})
print('Member 1 outputs are ready in:', PROJECT)


## 14. حفظ النتائج
هذه حزمة نتائج تحتوي صور الفريق وملفات التقسيم والتدقيق، وليست شرطًا لتشغيل كود هذا الدفتر. لا تنشرها في GitHub. في Colab جديد، ارفع ZIP من الشريط الجانبي ثم استخدم خلية الاستيراد في دفتر العضو 2 أو 3. إن كانت الجلسة نفسها، استخدم المجلد الموجود مباشرة. احفظ الدفتر بمخرجاته أيضًا.

In [ ]:
import shutil
archive = shutil.make_archive(str(PROJECT.parent/'Member1_Results'), 'zip', root_dir=PROJECT)
print('Saved:', archive)
from google.colab import files
files.download(archive)
